# 🚀 TRẢI NGHIỆM & THỬ NGHIỆM MÔ HÌNH FINE-TUNED QWEN2-VL TRÊN GPU TESLA T4
## Hệ thống Document Visual Question Answering (DocVQA) Hóa đơn Tiếng Việt

Notebook này cho phép bạn:
1. ⚡ Nạp trực tiếp mô hình **Qwen2-VL-2B + LoRA Adapter đã Fine-tune** trên GPU Tesla T4 (16GB VRAM).
2. 🔍 Hỏi đáp tự động trên các mẫu hóa đơn có sẵn (Viettel, VNPT, WinMart, Highlands Coffee, Phúc Long, KFC...).
3. 📤 Kéo thả ảnh hóa đơn bất kỳ từ máy tính của bạn và hỏi đáp tức thì qua giao diện **Gradio Web App**!

In [ ]:
# ==============================================================================
# BƯỚC 1: CÀI ĐẶT MÔI TRƯỜNG TƯƠNG THÍCH GPU TESLA T4
# ==============================================================================
print("📦 Đang cài đặt thư viện...")
!pip uninstall -y -q torchao
!pip install -q --no-deps qwen-vl-utils==0.0.8
!pip install -q "transformers==4.46.2" "peft==0.13.2" "accelerate==0.34.2" gradio pillow torchvision

import os
import sys
import gc
import time
import json
import zipfile
from pathlib import Path

import torch
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info

print(f"🔥 Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")
print(f"🧠 Tổng VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# ==============================================================================
# BƯỚC 2: TÌM & NẠP TRỌNG SỐ LORA ADAPTER ĐÃ HUẤN LUYỆN
# ==============================================================================
print("🔍 Đang tìm kiếm trọng số LoRA Adapter...")

adapter_dir = None
for root, dirs, files in os.walk("/kaggle"):
    if "adapter_model.safetensors" in files:
        adapter_dir = root
        print(f"✅ Tìm thấy LoRA Adapter tại: {adapter_dir}")
        break
    for f in files:
        if f.endswith(".zip") and "lora" in f.lower():
            extract_to = "/kaggle/working/loaded_lora"
            os.makedirs(extract_to, exist_ok=True)
            with zipfile.ZipFile(os.path.join(root, f), 'r') as zf:
                zf.extractall(extract_to)
            adapter_dir = os.path.join(extract_to, "qwen2_vl_lora_adapters") if os.path.exists(os.path.join(extract_to, "qwen2_vl_lora_adapters")) else extract_to
            print(f"✅ Đã giải nén LoRA Adapter vào: {adapter_dir}")
            break
    if adapter_dir:
        break

model_id = "Qwen/Qwen2-VL-2B-Instruct"
print(f"🧠 Đang nạp Base Model: {model_id} (FP16)... ")
processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=768*28*28)
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

if adapter_dir and os.path.exists(adapter_dir):
    print(f"🎯 Đang kích hoạt LoRA Adapter từ {adapter_dir}...")
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    print("🎉 KÍCH HOẠT LORA ADAPTER THÀNH CÔNG! Mô hình đã sẵn sàng suy luận!")
else:
    model = base_model
    print("⚠️ Không tìm thấy thư mục adapter, đang chạy ở chế độ Base Model.")

model.eval()


In [ ]:
# ==============================================================================
# BƯỚC 3: HÀM SUY LUẬN BÓC TÁCH THÔNG TIN HÓA ĐƠN
# ==============================================================================
def ask_receipt(image_input, question: str):
    """
    image_input: đường dẫn file ảnh (str) hoặc PIL Image
    question: câu hỏi tiếng Việt cần bóc tách
    """
    if isinstance(image_input, str):
        image = Image.open(image_input).convert("RGB")
    else:
        image = image_input.convert("RGB")
        
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]
    
    t0 = time.time()
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False
        )
        trimmed_generated_ids = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            trimmed_generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0].strip()
        
    latency = time.time() - t0
    return output_text, latency

print("✅ Hàm ask_receipt() đã sẵn sàng!")


In [ ]:
# ==============================================================================
# BƯỚC 4: THỬ NGHIỆM TRỰC TIẾP TRÊN CÁC MẪU HÓA ĐƠN CÓ SẴN
# ==============================================================================
sample_images = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            sample_images.append(os.path.join(root, f))
        if len(sample_images) >= 10:
            break
    if len(sample_images) >= 10:
        break

if sample_images:
    test_img = sample_images[0]
    print(f"🖼️ Thử nghiệm trên ảnh: {test_img}")
    
    questions = [
        "Tên đơn vị / người bán hàng trên hóa đơn là gì?",
        "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?",
        "Ngày giờ lập hóa đơn là khi nào?",
        "Địa chỉ của đơn vị bán hàng là ở đâu?"
    ]
    
    print("=" * 80)
    for q in questions:
        ans, lat = ask_receipt(test_img, q)
        print(f"❓ Câu hỏi : {q}")
        print(f"💡 Trả lời : {ans}")
        print(f"⏱️ Tốc độ  : {lat:.2f} giây")
        print("-" * 80)
else:
    print("Chưa nạp ảnh mẫu trong /kaggle/input, bạn có thể upload ảnh tự chọn ở Bước 5!")


In [ ]:
# ==============================================================================
# BƯỚC 5: KHỞI CHẠY GIAO DIỆN WEB GRADIO (KÉO THẢ ẢNH BẤT KỲ & CHAT TRỰC TIẾP)
# ==============================================================================
import gradio as gr

def gradio_interface(image, question):
    if image is None:
        return "Vui lòng tải lên một hình ảnh hóa đơn.", "0.00s"
    if not question.strip():
        return "Vui lòng nhập câu hỏi cần bóc tách.", "0.00s"
    ans, lat = ask_receipt(image, question)
    return ans, f"{lat:.2f} giây (GPU Tesla T4)"

with gr.Blocks(title="Document VQA - Hệ Thống Bóc Tách Hóa Đơn") as demo:
    gr.Markdown("# 🧾 Document VQA Demo: Qwen2-VL-2B + LoRA (Fine-Tuned)")
    gr.Markdown("Tải lên ảnh hóa đơn bất kỳ và đặt câu hỏi tiếng Việt để mô hình bóc tách kết quả trực tiếp.")
    
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Ảnh hóa đơn")
            question_input = gr.Textbox(
                label="Câu hỏi",
                placeholder="Ví dụ: Tổng tiền thanh toán là bao nhiêu?",
                value="Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?"
            )
            btn = gr.Button("🔍 Bóc Tách Thông Tin", variant="primary")
            
            gr.Markdown("### 💡 Câu hỏi gợi ý nhanh:")
            gr.Examples(
                examples=[
                    ["Tên đơn vị / người bán hàng trên hóa đơn là gì?"],
                    ["Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?"],
                    ["Ngày giờ lập hóa đơn là khi nào?"],
                    ["Địa chỉ của đơn vị bán hàng là ở đâu?"],
                    ["Mã số thuế của đơn vị bán hàng là gì?"]
                ],
                inputs=[question_input]
            )
            
        with gr.Column(scale=1):
            answer_output = gr.Textbox(label="Kết quả bóc tách từ Mô hình (Entity Value)", lines=4)
            latency_output = gr.Textbox(label="Độ trễ suy luận")
            
    btn.click(fn=gradio_interface, inputs=[image_input, question_input], outputs=[answer_output, latency_output])

print("🌐 Đang khởi chạy giao diện Gradio Demo...")
demo.launch(share=True, debug=False)
